# Exploratory Data Analysis

## Objective
Understand the structure, quality, and business patterns in the validated enterprise datasets before feature engineering and model development.

This notebook uses the reusable analytics engine in `backend/app/analytics/eda.py` and saves exportable analytical artifacts for the backend.

## Business Context

Executive intelligence depends on reliable data narratives: revenue trends, product performance, regional differences, customer segmentation, and seasonal behavior. The EDA layer turns validated tables into those narratives.

## Architecture and Implementation Plan

1. Load the processed datasets produced by the cleaning and validation stages.
2. Run summary statistics, missing-value analysis, and correlation analysis.
3. Analyze revenue, products, regions, seasonality, customer segments, and enterprise KPIs.
4. Persist plots and CSV artifacts into `exports/` and `reports/`.
5. Review the findings and capture conclusions that will guide feature engineering.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
backend_root = project_root / 'backend'
sys.path.insert(0, str(backend_root))

from app.analytics.eda import EDAPaths, EnterpriseEDA

processed_dir = project_root / 'processed'
reports_dir = project_root / 'reports'
exports_dir = project_root / 'exports'
eda = EnterpriseEDA(EDAPaths(processed_dir=processed_dir, reports_dir=reports_dir, exports_dir=exports_dir))
eda

## Load Summary Tables

Read the processed enterprise tables to inspect the business context and verify that the data is ready for analysis.

In [ ]:
tables = {
    'customers': pd.read_csv(processed_dir / 'customers.csv'),
    'products': pd.read_csv(processed_dir / 'products.csv'),
    'suppliers': pd.read_csv(processed_dir / 'suppliers.csv'),
    'employees': pd.read_csv(processed_dir / 'employees.csv'),
    'marketing_campaigns': pd.read_csv(processed_dir / 'marketing_campaigns.csv'),
    'orders': pd.read_csv(processed_dir / 'orders.csv'),
    'inventory_snapshots': pd.read_csv(processed_dir / 'inventory_snapshots.csv'),
    'finance_monthly': pd.read_csv(processed_dir / 'finance_monthly.csv'),
    'operations_daily': pd.read_csv(processed_dir / 'operations_daily.csv'),
    'customer_kpis': pd.read_csv(processed_dir / 'customer_kpis.csv'),
}
overview = pd.DataFrame({
    'table': list(tables.keys()),
    'rows': [len(frame) for frame in tables.values()],
    'columns': [len(frame.columns) for frame in tables.values()],
})
overview.sort_values('rows', ascending=False)

## Summary Statistics

Inspect the order table because it drives revenue, demand, and customer behavior analysis.

In [ ]:
summary_statistics = eda.summary_statistics(tables['orders'])
summary_statistics.head(20)

## Missing-Value Analysis

Identify any gaps that could distort downstream analytics or model training.

In [ ]:
missing_values = eda.missing_value_analysis(tables)
missing_values.head(25)

## Correlation Analysis

Look for relationships between commercial, customer, and product variables that can guide feature engineering.

In [ ]:
analysis_frame = eda._build_analysis_frame(
    customers=tables['customers'],
    products=tables['products'],
    orders=tables['orders'],
    customer_kpis=tables['customer_kpis'],
)
correlation = eda.correlation_analysis(analysis_frame)
correlation.select_dtypes(include='number').head() if hasattr(correlation, 'select_dtypes') else correlation.head()

## Revenue Analysis

Review the monthly revenue trajectory and the average order value trend.

In [ ]:
revenue_results = eda.revenue_analysis(tables['orders'])
revenue_results['monthly_revenue'].head(12)

## Product Analysis

Rank products by revenue and review the strongest commercial performers.

In [ ]:
product_summary = eda.product_analysis(tables['orders'], tables['products'])
product_summary.head(15)

## Regional Analysis

Compare revenue concentration across regions and customer segments.

In [ ]:
regional_summary = eda.regional_analysis(tables['customers'], tables['orders'])
regional_summary.head(20)

## Seasonal Trends

Inspect quarter-over-quarter and month-level seasonality in revenue and order volume.

In [ ]:
seasonal_summary = eda.seasonal_trend_analysis(tables['orders'])
seasonal_summary.head(20)

## Customer Segmentation Insights

Review how enterprise, mid-market, and SMB customers differ in revenue contribution and churn characteristics.

In [ ]:
segmentation_insights = eda.customer_segmentation_insights(tables['customer_kpis'])
segmentation_insights

## Business KPI Summary

Combine finance, operations, and customer health signals into a compact executive KPI summary.

In [ ]:
kpi_summary = eda.business_kpi_summary(tables['finance_monthly'], tables['operations_daily'], tables['customer_kpis'])
kpi_summary

## Visualization Preview

Show a compact set of visuals from the generated exports so the notebook remains publication-ready.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(tables['orders']['order_amount'], bins=40, kde=True, ax=axes[0], color='#1f77b4')
axes[0].set_title('Order Amount Distribution')
axes[0].set_xlabel('Order Amount')
sns.scatterplot(data=tables['customer_kpis'], x='total_orders', y='total_revenue', hue='segment', ax=axes[1])
axes[1].set_title('Orders vs Revenue by Customer Segment')
axes[1].set_xlabel('Total Orders')
axes[1].set_ylabel('Total Revenue')
plt.tight_layout()
plt.show()

## Conclusions

The validated enterprise data shows revenue concentration across a subset of products and customer segments, with clear monthly revenue signals and operational KPIs suitable for predictive modeling. The exported EDA artifacts in `exports/` and `reports/` are ready for backend and frontend consumption.